# THGS D — 시각 정성 확인

한 프레임에서 D arm 의 **인스턴스 분리** 가 눈으로 보기에 잘 됐는지만 본다.
GT, IoU, 라벨, 메트릭 다 빼고 4 패널 그림 한 장:

```
[입력 이미지]  [FastSAM 마스크]  [Feature PCA→RGB]  [Cluster 색칠]
                                       ↑                  ↑
                                같은 색 = 같은 의미       같은 색 = 같은 그룹
```

다른 객체들이 다른 색이면 D 가 인스턴스를 잘 분리한 것.
다른 객체가 같은 색으로 묶이면 D 가 의미적으로 혼동한 것.


In [ ]:
# 환경 + 데이터 + ckpt 준비
!nvidia-smi -L
import os, sys, shutil, zipfile
import numpy as np, cv2, torch
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/BAEJUNHAK/THGS.git"
WORK_DIR = "/content/THGS"
if not os.path.exists(WORK_DIR):
    !git clone --recursive {REPO_URL} {WORK_DIR}
os.chdir(WORK_DIR); !git pull

# 의존성
!pip install -q --upgrade opencv-python
!pip install -q omegaconf open_clip_torch ultralytics scikit-learn gdown

sys.path.insert(0, '/content/THGS')
sys.path.insert(0, '/content/THGS/scripts')

from omegaconf import OmegaConf
cfg = OmegaConf.load("configs/lerf.yml")
DATA_PATH = cfg.dataset.data_path
SCENE = "ramen"
SAMPLE_FRAME = "frame_00006"        # 다른 frame 보고 싶으면 여기만 수정

# LERF-OVS 데이터 (Drive 캐시 → gdown)
IMG_PATH = f"{DATA_PATH}/{SCENE}/images/{SAMPLE_FRAME}.jpg"
if not os.path.exists(IMG_PATH):
    drive_data = "/content/drive/MyDrive/THGS_cache/lerf_ovs"
    if os.path.exists(drive_data):
        os.makedirs("data", exist_ok=True)
        if not os.path.exists("data/lerf_ovs"):
            shutil.copytree(drive_data, "data/lerf_ovs")
        if not os.path.exists(DATA_PATH) and os.path.exists("data/lerf_ovs"):
            os.symlink("lerf_ovs", DATA_PATH)
    else:
        import gdown
        gdown.download(id="1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt",
                       output="/content/lerf_ovs.zip", quiet=False)
        with zipfile.ZipFile("/content/lerf_ovs.zip", 'r') as z:
            z.extractall("data/")
        os.remove("/content/lerf_ovs.zip")
        if not os.path.exists(DATA_PATH) and os.path.exists("data/lerf_ovs"):
            os.symlink("lerf_ovs", DATA_PATH)
assert os.path.exists(IMG_PATH), f"이미지 없음: {IMG_PATH}"

# FastSAM ckpt
FASTSAM_CKPT = "ckpts/FastSAM-x.pt"
os.makedirs("ckpts", exist_ok=True)
if not os.path.exists(FASTSAM_CKPT):
    try:
        from ultralytics import FastSAM
        m = FastSAM('FastSAM-x.pt')
        for c in ['FastSAM-x.pt',
                  os.path.expanduser('~/.config/Ultralytics/FastSAM-x.pt')]:
            if c and os.path.exists(c):
                shutil.copy2(c, FASTSAM_CKPT); break
        del m
    except Exception:
        pass
if not os.path.exists(FASTSAM_CKPT):
    !wget -q -O {FASTSAM_CKPT} \
        https://github.com/ultralytics/assets/releases/download/v8.2.0/FastSAM-x.pt
assert os.path.getsize(FASTSAM_CKPT) > 10_000_000, "FastSAM 다운로드 실패"
print(f"OK — image={IMG_PATH}, ckpt={FASTSAM_CKPT}")


In [ ]:
# D 파이프라인 실행: FastSAM 마스크 → ConvNeXt-L dense → mask 평균 풀링
from encode_fastsam_clip import fastsam_4scale, masks_update, LEVEL_NAMES
from encode_maskadapter_fastsam import ConvNeXtLDense
from encode_openseg_fastsam import avg_pool_per_mask
from ultralytics import FastSAM

image_bgr = cv2.imread(IMG_PATH)
H, W = image_bgr.shape[:2]
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# FastSAM 4-conf + NMS
print("FastSAM 마스크 생성...")
fastsam = FastSAM(FASTSAM_CKPT)
mp = fastsam_4scale(fastsam, image_bgr)
masks_d, masks_s, masks_m, masks_l = (mp[k] for k in LEVEL_NAMES)
masks_d, masks_s, masks_m, masks_l = masks_update(
    masks_d, masks_s, masks_m, masks_l,
    iou_thr=0.8, score_thr=0.7, inner_thr=0.5)
all_masks = []
for masks in [masks_d, masks_s, masks_m, masks_l]:
    all_masks.extend(masks)
print(f"  → {len(all_masks)} masks")

# ConvNeXt-L dense + mask pool
print("ConvNeXt-L dense feature + mask 평균 풀링...")
encoder = ConvNeXtLDense()
pixel_feat = encoder(image_rgb)         # (Hp, Wp, 768)
print(f"  dense feature: {pixel_feat.shape}")

emb, _ = avg_pool_per_mask(pixel_feat, all_masks, target_HW=(H, W))
feat = emb.float().numpy()              # (M, 768) L2-normalized
print(f"  per-mask feature: {feat.shape}")


In [ ]:
# 시각화 — 4 패널
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering

K_CLUSTERS = 10        # cluster 개수 (의미 그룹 수 추정 — 경험적으로 라멘 씬은 8~12)

assert len(feat) >= 1, "FastSAM 이 마스크를 0개 만듦 — 종료"

# Panel 3: feature → PCA(3) → RGB  (M<3 일 때는 그냥 임의 색)
if len(feat) >= 3:
    proj = PCA(n_components=3).fit_transform(feat)
    proj = (proj - proj.min(0)) / (proj.max(0) - proj.min(0) + 1e-9)
    mask_pca_color = proj          # (M, 3)
else:
    mask_pca_color = np.array([[1.0, 0.5, 0.5]] * len(feat))
    print(f"[경고] mask 가 {len(feat)}개뿐이라 PCA 생략 (회색 단색 표시)")

# Panel 4: cluster — K 는 mask 수를 넘을 수 없음
K_eff = min(K_CLUSTERS, len(feat))
if K_eff >= 2:
    try:
        cl = AgglomerativeClustering(n_clusters=K_eff, metric='cosine', linkage='average')
    except TypeError:
        cl = AgglomerativeClustering(n_clusters=K_eff, affinity='cosine', linkage='average')
    cluster_pred = cl.fit_predict(feat)
else:
    cluster_pred = np.zeros(len(feat), dtype=int)
cmap = plt.cm.get_cmap('tab20', max(K_eff, 1))
mask_cluster_color = np.array([cmap(c)[:3] for c in cluster_pred])

# 오버레이 헬퍼
def overlay(image_rgb_uint8, masks, colors, alpha=0.65):
    out = image_rgb_uint8.copy().astype(np.float32) / 255.0
    for m, color in zip(masks, colors):
        seg = m['segmentation']
        out[seg] = (1 - alpha) * out[seg] + alpha * np.asarray(color)[:3]
    return np.clip(out, 0, 1)

# Panel 2: random color per mask (FastSAM 자체)
rng = np.random.RandomState(42)
random_colors = rng.rand(len(all_masks), 3)

fig, axes = plt.subplots(1, 4, figsize=(24, 7))
axes[0].imshow(image_rgb);                                 axes[0].set_title("① 입력 이미지", fontsize=13)
axes[1].imshow(overlay(image_rgb, all_masks, random_colors)); axes[1].set_title(f"② FastSAM 마스크 (랜덤 색)\nN={len(all_masks)}", fontsize=13)
axes[2].imshow(overlay(image_rgb, all_masks, mask_pca_color));   axes[2].set_title("③ ConvNeXt-L feature → PCA→RGB\n같은 색 = 같은 의미", fontsize=13)
axes[3].imshow(overlay(image_rgb, all_masks, mask_cluster_color));axes[3].set_title(f"④ Cluster (K={K_eff})\n같은 색 = 같은 그룹", fontsize=13)
for ax in axes: ax.axis('off')

OUT = "output/D_visual"; os.makedirs(OUT, exist_ok=True)
plt.tight_layout()
plt.savefig(f"{OUT}/{SAMPLE_FRAME}_visual.png", dpi=130, bbox_inches='tight')
plt.show()
print(f"saved: {OUT}/{SAMPLE_FRAME}_visual.png")

print('''
직관 가이드:
  ② FastSAM 이 객체를 잘 짤랐는지 (놓친 객체 / 너무 잘게 쪼갬 등)
  ③ 객체별로 다른 색이 떠야 정상. 다른 객체끼리 같은 색이면 ConvNeXt-L 의미 혼동.
  ④ 한 cluster 가 한 객체에 깔끔히 매핑되면 분리 잘됨. 한 cluster 가 여러 객체를 묶으면 분리 실패.
  K_CLUSTERS 를 올리면 더 세분화, 내리면 더 거친 그룹.
''')
